# PDEep/simple in Google Colab
Run the existing CLI to generate numerical PDE data, train A-Geo/C-Geo with and without flow matching, and compare classical baselines.

Select **Runtime → Change runtime type → GPU** before starting. Work on Colab's local disk; Drive is optional. The defaults are a modest starting experiment, not a claim of useful reconstruction accuracy or calibrated uncertainty.

Run setup/settings first, then the cells you need. Each model has its own visible training and evaluation cell. If you skip a model, omit its directory from the comparison commands.


## 1. Clone and install
The repository's tracked root contains pyproject.toml and pdeep_simple/. Clone once in a fresh runtime, then install there. The editable install reuses compatible installed dependencies, including Colab's CUDA-enabled PyTorch; there is no separate torch install or CPU-wheel command.


In [ ]:
%cd /content
!git clone https://github.com/NiciiA/PDEep-simple.git
%cd /content/PDEep-simple
%pip install -e .


## 2. Check the GPU


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("Select Runtime → Change runtime type → GPU, reconnect, and rerun this cell.")
assert torch.cuda.is_available(), "A CUDA GPU is required for the neural training cells."
print("GPU:", torch.cuda.get_device_name(0))


## 3. Experiment settings
Counts are **per PDE–geometry combination**. These defaults give 96 training, 12 validation and 12 test cases. Edit the values for larger experiments.

Use a new data_dir/run_prefix for a different experiment, or the normal resume command for an existing run. Set dataset_config to "configs/dataset.toml" to use the optional generation settings; it is not passed to training. For T2 coefficient supervision, loss can include aux=.1; add other losses only where the chosen PDE supports them.


In [ ]:
from pathlib import Path

tier = "G1"
pdes = "poisson"
geometries = "square,disk,lshape"
train_cases, val_cases, test_cases = 32, 4, 4
dataset_config = None  # Or "configs/dataset.toml"

model_size = "s"
updates = 300
ae_updates = 100  # Included in updates: 100 representation + 200 flow updates.
batch = 2
queries = 512
latent_dim = 16
loss = "field=1"
modes = "forward,inverse,hybrid"
observation_counts = "0,8,16,32,64,128"
samples, flow_steps = 8, 8  # Increase, e.g. to 16/16, for a larger sampling experiment.
seed, noise = 42, 0.01

data_dir = "data/colab_g1"
run_prefix = "runs/colab"
baseline_methods = "prior,rbf,bayes,pod,gp"


## 4. Optional: mount Google Drive
Leave use_drive=False to run entirely on local disk. Mounting does not move training onto Drive. The final cells copy completed data/results, or restore them locally.


In [ ]:
use_drive = False
drive_dir = Path("/content/drive/MyDrive/PDEep-simple")

if use_drive:
    from google.colab import drive
    drive.mount("/content/drive")


## 5. Generate the dataset (CPU)
Both branches call the existing generator with the same experiment variables. Numerical FEM generation uses CPU workers, even though neural training uses CUDA. Omitted numerical settings retain their existing defaults.


In [ ]:
if dataset_config is None:
    !python -m pdeep_simple data --tier {tier} --pdes {pdes} \
        --geometries {geometries} --train {train_cases} --val {val_cases} --test {test_cases} \
        --workers auto --seed {seed} --out {data_dir}
else:
    !python -m pdeep_simple data --config {dataset_config} --tier {tier} --pdes {pdes} \
        --geometries {geometries} --train {train_cases} --val {val_cases} --test {test_cases} \
        --workers auto --seed {seed} --out {data_dir}


## 6. Inspect numerical PDE cases


In [ ]:
!python -m pdeep_simple inspect --data {data_dir} --split train --random 3 \
    --seed {seed} --observations 16 --noise {noise} --out {run_prefix}_inspection

from IPython.display import Image, display

for image_path in sorted(Path(f"{run_prefix}_inspection").glob("train_*.png"))[:3]:
    display(Image(filename=str(image_path)))


## 7. Train deterministic A-Geo
All four commands use the same data, size, modes, field loss, total update budget, batch and query settings. GPU availability is required; there is no CPU fallback. Validation is less frequent here to keep the example modest.


In [ ]:
assert torch.cuda.is_available(), "Run the GPU check cell before training."
!python -m pdeep_simple train --data {data_dir} --model A --size {model_size} \
    --modes {modes} --loss {loss} --updates {updates} --batch {batch} --queries {queries} \
    --observations {observation_counts} --noise {noise} --seed {seed} \
    --validate-every 50 --device cuda --run {run_prefix}_a


## 8. Train deterministic C-Geo


In [ ]:
assert torch.cuda.is_available(), "Run the GPU check cell before training."
!python -m pdeep_simple train --data {data_dir} --model C --size {model_size} \
    --modes {modes} --loss {loss} --updates {updates} --batch {batch} --queries {queries} \
    --observations {observation_counts} --noise {noise} --seed {seed} \
    --validate-every 50 --device cuda --run {run_prefix}_c


## 9. Train A-Geo + Flow
The first ae_updates learn the field representation. The remaining updates train the conditional latent flow with that representation frozen. **updates is the total across both stages**, so ae_updates must be smaller. Flow samples are learned conditional reconstructions, not automatically calibrated Bayesian samples.


In [ ]:
assert torch.cuda.is_available(), "Run the GPU check cell before training."
!python -m pdeep_simple train --data {data_dir} --model A --size {model_size} --flow \
    --latent-dim {latent_dim} --ae-updates {ae_updates} --updates {updates} \
    --modes {modes} --loss {loss} --batch {batch} --queries {queries} \
    --observations {observation_counts} --noise {noise} --seed {seed} \
    --flow-steps {flow_steps} --validate-every 50 --device cuda --run {run_prefix}_a_flow


## 10. Train C-Geo + Flow


In [ ]:
assert torch.cuda.is_available(), "Run the GPU check cell before training."
!python -m pdeep_simple train --data {data_dir} --model C --size {model_size} --flow \
    --latent-dim {latent_dim} --ae-updates {ae_updates} --updates {updates} \
    --modes {modes} --loss {loss} --batch {batch} --queries {queries} \
    --observations {observation_counts} --noise {noise} --seed {seed} \
    --flow-steps {flow_steps} --validate-every 50 --device cuda --run {run_prefix}_c_flow


## 11. Classical baselines (CPU)
Use the A run's training normalization for matching error units. If you skipped A, point normalization-run to another trained run on this dataset.

Prior uses available physics without solution observations; RBF/POD/GP use observations, with POD fitted on training fields. Bayes uses a supported linear-Gaussian PDE model. Unsupported combinations are recorded in skipped.csv: in particular, POD usually cannot match randomly resized G1 domains, and GP/Bayes have restricted PDE support. No fallback estimator is substituted.


In [ ]:
!python -m pdeep_simple baseline --data {data_dir} --split test --methods {baseline_methods} \
    --modes {modes} --observations {observation_counts} --noise {noise} --seed {seed} \
    --workers auto --device cpu --normalization-run {run_prefix}_a --out {run_prefix}_baselines

skipped_path = Path(f"{run_prefix}_baselines/skipped.csv")
if skipped_path.exists() and skipped_path.stat().st_size:
    print("Skipped combinations (first 12 lines):")
    print("".join(skipped_path.read_text().splitlines(keepends=True)[:12]))


### Optional: small MAP experiment
Leave run_map=False during the basic workflow. MAP repeatedly solves the PDE, so limit the number of cases. It supports small stationary source parameterizations, T2 scalars and configured T3 smooth coefficients, with at most 16 free parameters. Exact unsupported Hybrid constraints are skipped. Check map_success and solve counts; MAP is a point estimate without posterior uncertainty.

You can instead append map to baseline_methods above. To compare this separate limited run, add its directory to a plot command's runs list; the existing plots use shared cases.


In [ ]:
run_map = False
if run_map:
    !python -m pdeep_simple baseline --data {data_dir} --split test --methods map \
        --modes {modes} --observations {observation_counts} --noise {noise} --seed {seed} \
        --workers auto --device cpu --max-cases 2 --map-maxiter 100 \
        --normalization-run {run_prefix}_a --out {run_prefix}_map


## 12. Evaluate the four models
Each cell uses the same held-out split, conditioning modes, observation counts, noise and seed. The saved training scale is reused. Flow means, sample std, intervals and coverage come from the existing evaluator; no custom metrics are calculated here.


### A-Geo evaluation


In [ ]:
!python -m pdeep_simple eval --data {data_dir} --run {run_prefix}_a --split test \
    --modes {modes} --observations {observation_counts} --noise {noise} --seed {seed} \
    --device cuda --out {run_prefix}_a/evaluation


### C-Geo evaluation


In [ ]:
!python -m pdeep_simple eval --data {data_dir} --run {run_prefix}_c --split test \
    --modes {modes} --observations {observation_counts} --noise {noise} --seed {seed} \
    --device cuda --out {run_prefix}_c/evaluation


### A-Geo + Flow evaluation


In [ ]:
!python -m pdeep_simple eval --data {data_dir} --run {run_prefix}_a_flow --split test \
    --modes {modes} --observations {observation_counts} --noise {noise} --seed {seed} \
    --samples {samples} --flow-steps {flow_steps} --save-samples 2 \
    --device cuda --out {run_prefix}_a_flow/evaluation


### C-Geo + Flow evaluation


In [ ]:
!python -m pdeep_simple eval --data {data_dir} --run {run_prefix}_c_flow --split test \
    --modes {modes} --observations {observation_counts} --noise {noise} --seed {seed} \
    --samples {samples} --flow-steps {flow_steps} --save-samples 2 \
    --device cuda --out {run_prefix}_c_flow/evaluation


### Optional: multiple query resolutions
This extra experiment is off by default and uses only two cases of the A run. A finer query grid does **not** make the stored FEM reference more accurate. Reference convergence can be studied separately with the existing refine option (levels 1–3, with an explicit case limit).


In [ ]:
run_multires = False
if run_multires:
    !python -m pdeep_simple eval --data {data_dir} --run {run_prefix}_a --split test \
        --modes {modes} --observations {observation_counts} --noise {noise} --seed {seed} \
        --grids 128,256,512 --max-cases 2 --chunk 512 --device cuda \
        --out {run_prefix}_a_multigrid


## 13. Training and error comparisons
These commands use saved logs/metrics. Flow-matching loss and normalized field MSE remain separate. Curves compare common cases, observation counts, physical times and reference levels.


In [ ]:
!python -m pdeep_simple plot --runs {run_prefix}_a {run_prefix}_c \
    {run_prefix}_a_flow {run_prefix}_c_flow --what losses --out {run_prefix}_losses

!python -m pdeep_simple plot --runs {run_prefix}_a {run_prefix}_c \
    {run_prefix}_a_flow {run_prefix}_c_flow {run_prefix}_baselines \
    --what comparisons --metric normalized_rmse --out {run_prefix}_comparisons

from IPython.display import Image, display

display(Image(filename=f"{run_prefix}_losses/loss_comparison.png"))
for image_path in sorted(Path(f"{run_prefix}_comparisons").glob("comparisons_*_u*.png"))[:3]:
    display(Image(filename=str(image_path)))


### Same-case reconstructions, conditioning and uncertainty
The first selected PDE/geometry are used below; edit these two values to inspect another generated family. The existing plot command produces the FEM reference, all supported predictions, absolute errors, Forward/Inverse/Hybrid differences, and flow uncertainty/individual samples. Wave velocity figures are also saved for wave cases. GP intervals assume its kernel, and learned flow uncertainty is not automatically calibrated.


In [ ]:
case_pde = pdes.split(",")[0]
case_geometry = geometries.split(",")[0]

!python -m pdeep_simple plot --runs {run_prefix}_a {run_prefix}_c \
    {run_prefix}_a_flow {run_prefix}_c_flow {run_prefix}_baselines \
    --data {data_dir} --split test --pde {case_pde} --geometry {case_geometry} --case 0 \
    --modes {modes} --observations 32 --noise {noise} --seed {seed} --grid 128 \
    --samples {samples} --flow-steps {flow_steps} --save-samples 2 \
    --what fields,conditioning,uncertainty --device cuda --out {run_prefix}_fields


In [ ]:
from IPython.display import Image, display

fields_dir = Path(f"{run_prefix}_fields")
prefix_name = Path(run_prefix).name
selected_images = [
    "solutions_forward_u.png",
    "solutions_inverse_u.png",
    "solutions_hybrid_u.png",
    f"{prefix_name}_a_A-Geo_conditioning_u.png",
    f"{prefix_name}_c_C-Geo_conditioning_u.png",
    f"{prefix_name}_a_flow_A-Flow_inverse_uncertainty_u.png",
    f"{prefix_name}_c_flow_C-Flow_inverse_uncertainty_u.png",
    f"{prefix_name}_baselines_gp_inverse_uncertainty_u.png",
]
for name in selected_images:
    image_path = fields_dir / name
    if image_path.exists():  # Some uncertainty methods are unsupported for the chosen PDE.
        display(Image(filename=str(image_path)))


## 14. Optional: save completed files to Drive
Local Colab files can disappear when the runtime ends. Copy important checkpoints and results before disconnecting. Enable the earlier Drive mount first, then set save_to_drive=True here. This is a manual copy, not background synchronization.

The dataset is saved under the Drive folder's data subdirectory. Local run directories whose names begin with run_prefix are copied under runs. Copies merge matching folders; choose a different drive_dir to keep separate experiments.


In [ ]:
import shutil

save_to_drive = False
if save_to_drive:
    assert use_drive, "Enable and run the optional Drive mount cell first."
    drive_dir.mkdir(parents=True, exist_ok=True)
    shutil.copytree(data_dir, drive_dir / "data" / Path(data_dir).name, dirs_exist_ok=True)
    for source in sorted(Path(run_prefix).parent.glob(f"{Path(run_prefix).name}*")):
        if source.is_dir():
            shutil.copytree(source, drive_dir / "runs" / source.name, dirs_exist_ok=True)
    print("Copied dataset and runs to", drive_dir)


### Optional: restore locally and resume
In a new runtime, run setup, the GPU check, settings and the Drive mount, then restore the files before retraining or evaluating. Keep the same data_dir/run_prefix, or point them at the restored locations. Resume remains the existing train command; the requested update count is a new total, not additional updates.


In [ ]:
import shutil

restore_from_drive = False
if restore_from_drive:
    assert use_drive, "Enable and run the optional Drive mount cell first."
    shutil.copytree(drive_dir / "data" / Path(data_dir).name, data_dir, dirs_exist_ok=True)
    for source in sorted((drive_dir / "runs").glob(f"{Path(run_prefix).name}*")):
        if source.is_dir():
            shutil.copytree(source, Path(run_prefix).parent / source.name, dirs_exist_ok=True)


In [ ]:
resume_training = False
resume_updates = updates + 100
if resume_training:
    assert torch.cuda.is_available(), "Run the GPU check cell before training."
    !python -m pdeep_simple train --data {data_dir} --resume {run_prefix}_a_flow/last.pt \
        --updates {resume_updates} --device cuda --run {run_prefix}_a_flow
